# 20-тарау. Қолжазба цифрларды тану: MNIST

*«Машина қалай үйренеді?» кітабына арналған блокнот. Әр ұяшықтың түсіндірмесі кітапта берілген.*

Ұяшықтарды жоғарыдан төмен қарай ретімен іске қосыңыз: **Shift + Enter**.

## 20.1. Жобаның мақсаты

## 20.2. 1-қадам. Деректерді дайындау

In [ ]:
import numpy as np
import keras
from keras import layers

keras.utils.set_random_seed(42)
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

x_train = (x_train / 255.0).reshape(-1, 28, 28, 1)   # (60000, 28, 28, 1)
x_test = (x_test / 255.0).reshape(-1, 28, 28, 1)
print(x_train.shape, x_test.shape)

## 20.3. 2-қадам. Модельді құрастыру

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, kernel_size=3, activation="relu"),
    layers.MaxPooling2D(pool_size=2),
    layers.Conv2D(64, kernel_size=3, activation="relu"),
    layers.MaxPooling2D(pool_size=2),
    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(10, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

## 20.4. 3-қадам. Оқыту

In [ ]:
history = model.fit(x_train, y_train, epochs=8, batch_size=128,
                    validation_split=0.1, verbose=0)
h = history.history
for e in range(len(h["loss"])):
    print(f"Эпоха {e + 1}: оқыту = {h['accuracy'][e]:.4f}, "
          f"валидация = {h['val_accuracy'][e]:.4f}")

## 20.5. 4-қадам. Тест және қателерді талдау

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Тест дәлдігі: {test_acc:.2%}")

probs = model.predict(x_test, verbose=0)
pred = probs.argmax(axis=1)
wrong = np.where(pred != y_test)[0]
print("10 000 цифрдың ішінде қате танылғаны:", len(wrong))

## 20.6. 5-қадам. Өз цифрыңызды тану

In [ ]:
from google.colab import files
files.upload()        # ашылған терезеден my_digit.png файлын таңдаңыз

In [ ]:
from PIL import Image, ImageDraw

canvas = Image.new("L", (280, 280), color=255)          # ақ фон
draw = ImageDraw.Draw(canvas)
points = [(60, 70), (205, 62), (150, 150), (115, 235)]   # «7» сызығы
draw.line(points, fill=0, width=26, joint="curve")
canvas.save("my_digit.png")

In [ ]:
from PIL import ImageOps

def prepare_image(path):
    img = Image.open(path).convert("L")          # сұр түске айналдыру
    img = ImageOps.invert(img)                   # ақ фон → қара фон
    img = img.crop(img.getbbox())                # цифрдың айналасын қиып алу
    img.thumbnail((20, 20))                      # 20×20-ға сыйдыру
    result = Image.new("L", (28, 28), color=0)   # 28×28 қара жақтау
    left = (28 - img.width) // 2
    top = (28 - img.height) // 2
    result.paste(img, (left, top))               # ортаға орналастыру
    return np.array(result).reshape(1, 28, 28, 1) / 255.0

x = prepare_image("my_digit.png")
p = model.predict(x, verbose=0)[0]
print(f"Желінің жауабы: {p.argmax()} (сенімділігі {p.max():.0%})")
top3 = np.argsort(p)[::-1][:3]
print("Үздік үш нұсқа:", [(int(d), f"{p[d]:.1%}") for d in top3])

## 20.7. 6-қадам. Модельді сақтау

In [ ]:
model.save("mnist_cnn.keras")
print("Модель сақталды!")